# 14.4 MLP activation 可以怎麼選？


假設一層模型先把「這裡需要某項特徵」算成數字，輸出可能是負的、零或正的。接下來應直接把它送出，還是改變不同大小的通過方式？如果各層都只有矩陣乘法，連續兩次線性轉換仍能合成一次線性轉換，增加層數也無法表達新的彎曲關係。[混合位置後處理特徵](https://birdhackor.github.io/tiny-perceptron-vlm/4.4.html)所用的MLP，就是在矩陣乘法之間加入非線性函數；activation（啟動函數）指的便是這個逐項改數字的步驟。

用三個容易追蹤的輸入 `[-1,0,2]` 看兩種選擇。ReLU先把負數改成零、保留正數，因此得到 `[0,0,2]`；ReLU²再平方，得到 `[0,0,4]`。GELU則平滑地縮小數字。它可寫成 `x × Φ(x)`，其中Φ是標準常態分布累積到x的機率，介於0和1。對x=-1，Φ約0.1587，輸出約-0.1587；對x=2，Φ約0.9772，輸出約1.9545。負值沒有整段硬切成零，而大正值接近原值。

下面擴成五個固定輸入，避免隨機值讓重點難辨。`F`是PyTorch提供常用函數的模組；`F.gelu`實作上述GELU，`F.relu(x).square()`清楚分成截掉負值、再平方兩步。所有函數都逐項工作，輸入與輸出形狀相同。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
import torch.nn.functional as F

x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
print("輸入", x.tolist())
print("GELU", F.gelu(x).round(decimals=4).tolist())
print("ReLU平方", F.relu(x).square().tolist())

GELU約輸出 `[-0.0455,-0.1587,0,0.8413,1.9545]`；ReLU²為 `[0,0,0,1,4]`。觀察x=-2與-1：GELU並不是把所有負值乘同一個常數，更大的負值反而更接近零。觀察正側：ReLU²在x=2時變成4，比x=1時的1大四倍，因此對大正值更敏感。這些數字說明函數如何處理訊號，還沒有說明它們在文字任務上誰比較好。

把啟動函數換到訓練好的模型，會改變下一層收到的尺度和負值分布。即使新函數不增加參數，也可能需要讓權重重新適應。公平比較應固定訓練資料、層數、特徵寬度與訓練步數，分別訓練後再比較驗證loss（未用於更新權重的資料上，預測答案的誤差）及耗時。單看函數名字、單次輸出或省一個運算，不能替整個模型排名。

練習把 `x` 改成 `[-4,-2,0,2,4]`。執行前先預測ReLU²最後一項是16，GELU最後一項接近4；再核對負側是否依然保留很小的負數。若畫圖，橫軸應是輸入，縱軸應是各函數輸出，不能把這條曲線當作訓練品質圖。
